# Volatility Forecasting: ML Training (Colab)

Re-runs Stage 7 of the volatility forecasting pipeline: five ML models (Lasso, Elastic Net, Random Forest, Gradient Boosting, NN_2 with 10-net ensemble) on three stocks (AAPL, JPM, AMZN) and two information sets (M_HAR and M_ALL).

The notebook downloads the pre-computed master DataFrames from GitHub, runs the models, and saves test-set predictions to a local `predictions/` folder. Prints a pivot MSE table at the end.

**Before running:** fill in `GITHUB_USER` and `GITHUB_REPO` in the next cell.

**Expected runtime:**
- With a Colab GPU (T4/A100/L4): roughly 5 to 15 minutes (NN ensemble dominates).
- CPU only: roughly 2 to 4 hours.


In [ ]:
# Fill these in with your GitHub username and repo name.
GITHUB_USER = "TODO"
GITHUB_REPO = "TODO"
BRANCH = "main"

TICKERS = ["AAPL", "JPM", "AMZN"]
PREDICTIONS_DIR = "predictions"
DATA_DIR = "data/final"


## Setup and GPU check

If a GPU is available, TensorFlow will pick it up automatically for the NN ensemble (final cell). The other four model families run on CPU regardless.


In [ ]:
import os
import random
import time
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import ElasticNetCV, LassoCV
from sklearn.model_selection import TimeSeriesSplit

# Seeds for reproducibility
np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

gpus = tf.config.list_physical_devices("GPU")
print(f"TensorFlow version: {tf.__version__}")
print(f"GPUs available: {gpus if gpus else 'none, will use CPU'}")

Path(PREDICTIONS_DIR).mkdir(exist_ok=True)
Path(DATA_DIR).mkdir(parents=True, exist_ok=True)


## Data loading

Downloads the three pre-computed master DataFrames (one per stock) from the GitHub raw URLs into a local `data/final/` folder. Files are cached, so re-running the cell does not re-download.

Each file contains the target column `RV` plus 12 predictor columns, indexed by date. Stages 1-4 of the pipeline produced these files; see the project README for details.


In [ ]:
RAW_URL_TEMPLATE = "https://raw.githubusercontent.com/{user}/{repo}/{branch}/data/final/master_{ticker}.csv"


def download_if_missing(ticker):
    local_path = Path(DATA_DIR) / f"master_{ticker}.csv"
    if local_path.exists():
        print(f"  {ticker}: cached at {local_path}")
        return local_path
    url = RAW_URL_TEMPLATE.format(user=GITHUB_USER, repo=GITHUB_REPO, branch=BRANCH, ticker=ticker)
    print(f"  {ticker}: downloading from {url}")
    urllib.request.urlretrieve(url, local_path)
    return local_path


HAR_FEATURES = ["RVD", "RVW", "RVM"]
M_ALL_EXTRAS = ["M1W", "d_log_dvol", "EA", "VIX", "d_US3M", "HSI", "ADS", "EPU"]
M_ALL_FEATURES = HAR_FEATURES + M_ALL_EXTRAS
TARGET_COL = "RV"
TRAIN_FRAC = 0.70
VAL_FRAC = 0.10
INFOSETS = ["M_HAR", "M_ALL"]
ALL_CELLS = [(t, i) for t in TICKERS for i in INFOSETS]


def split_stock(ticker):
    """Chronological 70/10/20 split with training-set standardisation (ddof=0)."""
    path = download_if_missing(ticker)
    df = pd.read_csv(path, parse_dates=["date"], index_col="date").sort_index()
    n = len(df)
    n_train = int(n * TRAIN_FRAC)
    n_val = int(n * VAL_FRAC)
    train = df.iloc[:n_train]
    val = df.iloc[n_train:n_train + n_val]
    test = df.iloc[n_train + n_val:]
    X_train = train.drop(columns=[TARGET_COL])
    y_train = train[TARGET_COL]
    X_val = val.drop(columns=[TARGET_COL])
    y_val = val[TARGET_COL]
    X_test = test.drop(columns=[TARGET_COL])
    y_test = test[TARGET_COL]
    train_mean = X_train.mean()
    train_std = X_train.std(ddof=0)
    return {
        "ticker": ticker,
        "X_train": X_train, "y_train": y_train,
        "X_val": X_val, "y_val": y_val,
        "X_test": X_test, "y_test": y_test,
        "X_train_std": (X_train - train_mean) / train_std,
        "X_val_std": (X_val - train_mean) / train_std,
        "X_test_std": (X_test - train_mean) / train_std,
    }


def features_for(info_set):
    return HAR_FEATURES if info_set == "M_HAR" else M_ALL_FEATURES


splits = {t: split_stock(t) for t in TICKERS}
for t, sp in splits.items():
    print(f"  {t}: train={len(sp['X_train'])}  val={len(sp['X_val'])}  test={len(sp['X_test'])}")

results = {}  # (ticker, info_set, model) -> dict


## Lasso and Elastic Net

Linear models with regularisation. `α` is chosen by 5-fold TimeSeriesSplit cross-validation on the training set. Elastic Net additionally selects `l1_ratio` from `[0.1, 0.5, 0.7, 0.9, 0.95, 0.99]`. Fast: a few seconds total across all 6 (stock × info-set) cells.


In [ ]:
def negative_clip(preds, y_train):
    """Replace any pred < 0 with min(y_train). Returns (preds, n_clipped)."""
    min_rv = float(y_train.min())
    n_neg = int((preds < 0).sum())
    if n_neg > 0:
        preds = np.where(preds < 0, min_rv, preds)
    return preds, n_neg


def fit_and_save(ticker, info_set, model_name, model, sp):
    """Fit a sklearn model, predict on test, apply negative-clip, save CSV, record MSE."""
    cols = features_for(info_set)
    X_train = sp["X_train_std"][cols].to_numpy()
    y_train = sp["y_train"].to_numpy()
    X_test = sp["X_test_std"][cols].to_numpy()
    y_test = sp["y_test"].to_numpy()
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    preds, n_neg = negative_clip(preds, sp["y_train"])
    mse = float(((preds - y_test) ** 2).mean())
    out_path = Path(PREDICTIONS_DIR) / f"{ticker}_{info_set}_{model_name}_h1.csv"
    pd.DataFrame({
        "actual": sp["y_test"],
        "predicted": pd.Series(preds, index=sp["y_test"].index),
    }).to_csv(out_path)
    results[(ticker, info_set, model_name)] = {"test_mse": mse, "n_neg": n_neg}
    return mse, n_neg


for ticker, info_set in ALL_CELLS:
    sp = splits[ticker]
    lasso = LassoCV(cv=TimeSeriesSplit(n_splits=5), random_state=42, max_iter=20000)
    mse, n_neg = fit_and_save(ticker, info_set, "Lasso", lasso, sp)
    print(f"  Lasso  {ticker:>5} x {info_set:<6} MSE={mse:.4e}  alpha={lasso.alpha_:.2e}  neg_clip={n_neg}")
    en = ElasticNetCV(
        l1_ratio=[0.1, 0.5, 0.7, 0.9, 0.95, 0.99],
        cv=TimeSeriesSplit(n_splits=5),
        random_state=42, max_iter=20000,
    )
    mse, n_neg = fit_and_save(ticker, info_set, "ElasticNet", en, sp)
    print(f"  EN     {ticker:>5} x {info_set:<6} MSE={mse:.4e}  alpha={en.alpha_:.2e}  l1={en.l1_ratio_:.2f}  neg_clip={n_neg}")


## Random Forest and Gradient Boosting

Tree models with default hyperparameters per Stage 7 spec. GB uses `max_depth=2`, matching the paper's Table A.6 tuning grid {1, 2}.

RF with 500 trees runs in a few seconds per cell (parallel via `n_jobs=-1`). GB runs sequentially and is slightly slower but still finishes inside a minute total.


In [ ]:
for ticker, info_set in ALL_CELLS:
    sp = splits[ticker]
    rf = RandomForestRegressor(
        n_estimators=500, max_features="sqrt", random_state=42, n_jobs=-1
    )
    mse, n_neg = fit_and_save(ticker, info_set, "RF", rf, sp)
    print(f"  RF     {ticker:>5} x {info_set:<6} MSE={mse:.4e}  neg_clip={n_neg}")
    gb = GradientBoostingRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=2, random_state=42
    )
    mse, n_neg = fit_and_save(ticker, info_set, "GB", gb, sp)
    print(f"  GB     {ticker:>5} x {info_set:<6} MSE={mse:.4e}  neg_clip={n_neg}")


## Neural network ensemble

Trains 100 NN_2 models per (stock × info-set) cell with seeds 0 to 99. For each cell, picks the top 10 nets by validation MSE and ensembles them by mean prediction. This is the most expensive step in the pipeline.

Architecture: `Dense(4) -> LeakyReLU(0.01) -> Dropout(0.2) -> Dense(2) -> LeakyReLU(0.01) -> Dropout(0.2) -> Dense(1, linear)`.

Optimiser: Adam with learning rate 0.001. Batch size 32. Max 500 epochs with `EarlyStopping(patience=20, restore_best_weights=True)` on validation MSE.

Estimated runtime: roughly 5 to 15 minutes on a Colab GPU, several hours on CPU.


In [ ]:
tf.get_logger().setLevel("ERROR")

NN_TOTAL = 100
NN_TOP = 10
NN_MAX_EPOCHS = 500
NN_PATIENCE = 20
NN_BATCH_SIZE = 32
NN_LR = 0.001
NN_DROPOUT = 0.2
NN_LRELU_ALPHA = 0.01


def make_nn(n_features):
    """Construct a fresh NN_2 model: 4-neuron then 2-neuron hidden layers with LeakyReLU and dropout."""
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(n_features,)),
        tf.keras.layers.Dense(4),
        tf.keras.layers.LeakyReLU(negative_slope=NN_LRELU_ALPHA),
        tf.keras.layers.Dropout(NN_DROPOUT),
        tf.keras.layers.Dense(2),
        tf.keras.layers.LeakyReLU(negative_slope=NN_LRELU_ALPHA),
        tf.keras.layers.Dropout(NN_DROPOUT),
        tf.keras.layers.Dense(1, activation="linear"),
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=NN_LR), loss="mse")
    return model


def fit_nn_ensemble(X_train, y_train, X_val, y_val):
    """Train NN_TOTAL nets with seeds 0..NN_TOTAL-1; return (nets, top_idx, val_mses)."""
    val_mses = []
    nets = []
    t0 = time.time()
    for seed in range(NN_TOTAL):
        tf.keras.backend.clear_session()
        tf.random.set_seed(seed)
        np.random.seed(seed)
        random.seed(seed)
        m = make_nn(X_train.shape[1])
        es = tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=NN_PATIENCE, restore_best_weights=True
        )
        m.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=NN_MAX_EPOCHS,
            batch_size=NN_BATCH_SIZE,
            callbacks=[es],
            verbose=0,
        )
        val_pred = m.predict(X_val, verbose=0).flatten()
        val_mses.append(float(((val_pred - y_val) ** 2).mean()))
        nets.append(m)
        if (seed + 1) % 10 == 0:
            print(f"    NN progress {seed + 1}/{NN_TOTAL}  elapsed={time.time() - t0:.1f}s")
    val_mses = np.array(val_mses)
    top_idx = np.argsort(val_mses)[:NN_TOP]
    return nets, top_idx, val_mses


for ticker, info_set in ALL_CELLS:
    sp = splits[ticker]
    cols = features_for(info_set)
    X_train = sp["X_train_std"][cols].to_numpy()
    y_train = sp["y_train"].to_numpy()
    X_val = sp["X_val_std"][cols].to_numpy()
    y_val = sp["y_val"].to_numpy()
    X_test = sp["X_test_std"][cols].to_numpy()
    y_test = sp["y_test"].to_numpy()
    print(f"\n  NN_2_e10 {ticker} x {info_set} (training {NN_TOTAL} nets)")
    nets, top_idx, val_mses = fit_nn_ensemble(X_train, y_train, X_val, y_val)
    preds_each = np.column_stack([nets[i].predict(X_test, verbose=0).flatten() for i in top_idx])
    preds = preds_each.mean(axis=1)
    preds, n_neg = negative_clip(preds, sp["y_train"])
    mse = float(((preds - y_test) ** 2).mean())
    out_path = Path(PREDICTIONS_DIR) / f"{ticker}_{info_set}_NN_2_e10_h1.csv"
    pd.DataFrame({
        "actual": sp["y_test"],
        "predicted": pd.Series(preds, index=sp["y_test"].index),
    }).to_csv(out_path)
    results[(ticker, info_set, "NN_2_e10")] = {
        "test_mse": mse, "n_neg": n_neg,
        "top10_val_mse_mean": float(val_mses[top_idx].mean()),
        "all100_val_mse_mean": float(val_mses.mean()),
        "all100_val_mse_std": float(val_mses.std()),
        "best_val": float(val_mses.min()),
        "worst_val": float(val_mses.max()),
        "top_seeds": top_idx.tolist(),
    }
    print(f"  NN_2_e10 {ticker:>5} x {info_set:<6} test MSE={mse:.4e}  top10 val MSE mean={val_mses[top_idx].mean():.4e}  neg_clip={n_neg}")

# Final diagnostics
print("\n" + "=" * 78)
print("  Test MSE pivot (per ticker x info-set x model)")
print("=" * 78)
df = pd.DataFrame([
    {"ticker": k[0], "info_set": k[1], "model": k[2], "test_mse": v["test_mse"]}
    for k, v in results.items()
])
pivot = df.pivot_table(index=["ticker", "model"], columns="info_set", values="test_mse")
print(pivot.to_string(float_format=lambda x: f"{x:.4e}"))

print("\n" + "=" * 78)
print("  NN ensemble diagnostics")
print("=" * 78)
for (ticker, info_set, model), v in results.items():
    if model != "NN_2_e10":
        continue
    print(f"  {ticker} x {info_set}:")
    print(f"    top-10 val MSE mean : {v['top10_val_mse_mean']:.4e}")
    print(f"    all-100 mean        : {v['all100_val_mse_mean']:.4e}    std: {v['all100_val_mse_std']:.4e}")
    print(f"    best / worst val MSE: {v['best_val']:.4e} / {v['worst_val']:.4e}")
    print(f"    top-10 seeds        : {v['top_seeds']}")
